# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout, stratified 5-fold CV on training rows, choice by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='083821b17b5c7fdf81de46633ca08a7d601bd8266b269faa3911938d26bf66bf'
df=pd.read_csv('data.csv'); target='claim_within_year'
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
train,hold=df.iloc[train_idx],df.iloc[hold_idx]
print(len(train),len(hold),train[target].mean(),hold[target].mean())

## Features
`device_serial` is excluded (identifier, per protocol). The other columns (channel, device age, price, reported daily use) are all recorded at sale/registration, i.e. known at prediction time.

In [ ]:
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
features=cat+num
X,y=train[features],train[target]
print(train[features].describe(include='all').T)
print(train.groupby('channel')[target].mean())

## Candidates (all named before any scoring)
Preprocessing lives inside the pipelines so it is fitted within each CV fold.

In [ ]:
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates={
 'baseline_prior': make_pipeline(pre(False),DummyClassifier(strategy='prior')),
 'logreg_C1': make_pipeline(pre(True),LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1': make_pipeline(pre(True),LogisticRegression(C=0.1,max_iter=1000)),
 'random_forest': make_pipeline(pre(False),RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0)),
 'hist_gb': make_pipeline(pre(False),HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
print(list(candidates))

In [ ]:
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=0)
folds=[(tr.tolist(),va.tolist()) for tr,va in skf.split(X,y)]
cv_scores={n:cross_val_score(m,X,y,cv=skf,scoring='roc_auc').tolist() for n,m in candidates.items()}
summary=pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T
print(summary.sort_values('mean',ascending=False))
chosen=summary['mean'].idxmax(); print('chosen:',chosen)

## Refit on all training rows, score once on holdout

In [ ]:
final=candidates[chosen].fit(X,y)
holdout_auc=roc_auc_score(hold[target],final.predict_proba(hold[features])[:,1])
print('holdout roc_auc',holdout_auc)

In [ ]:
# row labels = 0-based positions in data.csv
cv_folds=[[train_idx[tr].tolist(),train_idx[va].tolist()] for tr,va in folds]
evidence={'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':hold_idx.tolist(),'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))